In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
# 1. Loading data
# Load wildfire data
fire_tot = pd.read_csv("../data/processed/fire_tot.csv", sep = ",")
print("Data loaded successfully")

# Load canada data
canada = gpd.read_file("../data/raw/lpr_000b21a_e.zip")
print("Zip loaded successfully")

In [ ]:
# 3. Rename "PRENAME" to "province" and export
# Rename
print(canada.columns)
canada = canada.rename(columns={"PRENAME": "province"})

# Export
canada.to_file("../data/processed/choropleth_map/canada_provinces.gpkg", driver="GPKG")
print("Export sucessful")

In [ ]:
# 4. Area of Canada in hectares
canada["area_ha"] = canada.geometry.area / 10000
print(canada[["province", "area_ha"]])

In [ ]:
# 5. Contolling if the files have the same provinces
set(canada["province"]) - set(fire_tot["province"])

In [ ]:
# 6. Calculate burnt area for each province for each year
fire_sum = ( fire_tot.groupby(["year", "province"])["size_ha"].sum().reset_index())

fire_sum.head(12)

In [ ]:
# 7. Merge together
fire_sum = fire_sum.merge(
    canada[["province", "area_ha"]],
    on="province",
    how="left" # keep all rows from fire_sum and merge the data that fits from canada to fire_sum
)

In [ ]:
# 8. Calculation of burnt area ratio for each province
fire_sum["burnt_percent"] = (fire_sum["size_ha"] / fire_sum["area_ha"]) * 100

fire_sum.head(20)

In [ ]:
# 9. Exporting
fire_sum.to_csv("../data/processed/choropleth_map/burnt_area.csv", index = False)
print("Export successful")